In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import os
import time
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, x_r, theta_0):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

def append_result_time(d_time, index, e_time):
    d_time["i"].append(index)
    d_time["e_time"].append(np.float64(e_time).round(5))

In [4]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset, base_model: NN, X_train):
    alpha = params['alpha']
    lamb = params['lamb']
    recourse_i = params['recourse_index']
    f_name = f'../results/recourse/nn_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl'
    f_name_time = f'../results/recourse_time/nn_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}_time.pkl'
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    results_time = {'i': [], "e_time": []}

    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)

    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]

        start_time = time.perf_counter()
        # LIME approximation of original NN
        np.random.seed(i)
        weights_0, bias_0 = lime_explanation(base_model.predict, X_train, x_0)
        weights_0, bias_0 = np.round(weights_0, 4), np.round(bias_0, 4)
        theta_0 = np.hstack((weights_0, bias_0))
        
        # Initalize recourse methods with theta_0
        recourse.set_weights(weights_0)
        recourse.set_bias(bias_0)

        x_r = recourse.get_recourse(x_0)
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time

        append_result(results, recourse.name, seed, alpha, lamb, recourse_i[i], x_0, x_r, theta_0)
        append_result_time(results_time, recourse_i[i], elapsed_time)

    df_results = pd.DataFrame(results)
    df_results_time = pd.DataFrame(results_time)

    if params['append_results'] and os.path.exists(f_name):
        df_tmp = pd.read_pickle(f_name)
        df_results = pd.concat((df_tmp, df_results), axis=0).sort_values(['i'], ignore_index=True)
    if params['append_results'] and params['save_time'] and os.path.exists(f_name_time):
        df_time_tmp = pd.read_pickle(f_name_time)
        df_results_time = pd.concat((df_time_tmp, df_results_time), axis=0).sort_values(['i'], ignore_index=True)

    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f_name)
    if params["save_time"]:
        print(f'[{recourse.name}] Saving time results for {dataset.name} run {seed}')
        df_results_time.to_pickle(f_name_time)
    
    return df_results

In [5]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = NN(X_train.shape[1])
        base_model.train(X_train.values, y_train.values)
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse_needed_X_test_idx = np.arange(recourse_needed_X_test.shape[0])

            recourse = recourse_fn(weights=None, bias=None, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            if params['append_results']:
                f_name = f"../results/recourse/nn_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
                if os.path.exists(f_name):
                    df_tmp = pd.read_pickle(f_name)
                    recourse_needed_X_test_idx = np.setdiff1d(recourse_needed_X_test_idx, df_tmp['i'].to_numpy())
                else:
                    print(f"{f_name} does not exist. The recourses will be created in a new file")

                if recourse_needed_X_test_idx.size == 0:
                    print(f"{f_name} already has all the recourses. Skipping")
                    continue
            
            if params['subsample']:
                rng = np.random.default_rng(seed=seed)
                size_N = int(np.rint(params['subsample_size'] * recourse_needed_X_test.shape[0]))
                if recourse_needed_X_test_idx.shape[0] < size_N:
                    size_N = recourse_needed_X_test_idx.shape[0]
                recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=size_N, replace=False)

            # if params['sba_fix']:
            #     prev_indexes = {0: np.array([24,19,31,0,2,1]),
            #                     1: np.array([17,16,27,29,34,4]),
            #                     2: np.array([9,4,30,29,3,17]),
            #                     3: np.array([2,6,27,30,21,26]),
            #                     4: np.array([26,33,34,35,2,36])}
                
            #     f_name = f"../results/recourse/nn_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
            #     if os.path.exists(f_name):
            #         df_tmp = pd.read_pickle(f_name)
            #         recourse_needed_X_test_idx = np.setdiff1d(prev_indexes[seed], df_tmp['i'].to_numpy())
            #     else:
            #         print("SOMETHING WENT WRONG")
            #         continue

            #     rng = np.random.default_rng(seed=seed)
            #     if recourse_needed_X_test_idx.size < 3:
            #         recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=recourse_needed_X_test_idx.size, replace=False)
            #     else:
            #         recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=3, replace=False)
            
            params['recourse_index'] = recourse_needed_X_test_idx.copy()
            recourse_needed_X_test = recourse_needed_X_test[recourse_needed_X_test_idx]
                
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset, base_model, X_train)
            results.append(df_results)

nn_sba(alpha=0.1) <br>
Seed0 - 24,19,31,0,2,1 <br>
Seed1 - 17,16,27,29,34,4 <br>
Seed2 - 9,4,30,29,3,17 <br>
Seed3 - 2,6,27,30,21,26 <br>
Seed4 - 26,33,34,35,2,36 <br>
<br>
nn_sba(alpha=0.5) <br>
Seed0 - 32,24 <br>
Seed1 - 16,18 <br>
Seed2 - 10,31 <br>
Seed3 - 3,28 <br>
Seed4 - 26,35 <br>

In [6]:
np.hstack((np.array([0.00001, 0.0001]), np.arange(0.001, 0.0105, 0.001), np.arange(0.02, 0.105, 0.01), np.arange(0.1, 1.05, 0.1))).round(7)

array([1.e-05, 1.e-04, 1.e-03, 2.e-03, 3.e-03, 4.e-03, 5.e-03, 6.e-03,
       7.e-03, 8.e-03, 9.e-03, 1.e-02, 2.e-02, 3.e-02, 4.e-02, 5.e-02,
       6.e-02, 7.e-02, 8.e-02, 9.e-02, 1.e-01, 1.e-01, 2.e-01, 3.e-01,
       4.e-01, 5.e-01, 6.e-01, 7.e-01, 8.e-01, 9.e-01, 1.e+00])

In [ ]:
alphas = [0.5] # <------------------------
lambdas = [0.3] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:
        
        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True
        params['append_results'] = False
        params['subsample'] = False
        params['subsample_size'] = 0.25
        params['save_time'] = True

        params['sba_fix'] = False

        datasets = [GermanDataset()] # <------------------------
        recourse_fns = [LARRecourse] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running sba data...


[L1PSD] [alpha=0.5] [lambda=0.1]:   0%|          | 0/3 [00:04<?, ?it/s]


KeyboardInterrupt: 

0 [24 19 31] [0 2 1]
1 [17 16 27] [29 34  4]
2 [ 9  4 30] [29  3 17]
3 [ 2  6 27] [30 21 26]
4 [26 33 34] [35  2 36]